In [1]:
from pathlib import Path
import numpy as np

# Import sparse Parflex implementation from the notebook module
from Sparse_Parflex import (
    align_system_sparse_parflex,
    chunked_flexdtw,
    sync_overlapping_positions,
    DEFAULT_CHUNK_LENGTH,
)

# Base directory where Mazurka feature .npy files live
MAZURKA_FEATURE_ROOT = Path("/home/ijain/ttmp/Chopin_Mazurkas_features/matching")

# Train queries file listing all Mazurka train pairs (both pieces)
TRAIN_QUERIES_PATH = Path("cfg_files/queries.train.full")

# Mapping from shorthand "train Mazurka #" to actual piece IDs
TRAIN_MAZURKAS = {
    "Mazurka_1": "Chopin_Op017No4",
    "Mazurka_2": "Chopin_Op063No3",
}

In [2]:
def _parse_train_queries(queries_path: Path):
    """Parse cfg_files/queries.train.full into a dict[piece] -> list[(id1, id2)]."""
    piece_to_pairs = {}
    with open(queries_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            left, right = line.split()
            piece1, fileid1 = left.split("/", 1)
            piece2, fileid2 = right.split("/", 1)
            assert piece1 == piece2, f"Mismatched pieces in line: {line}"
            piece = piece1
            piece_to_pairs.setdefault(piece, []).append((fileid1, fileid2))
    return piece_to_pairs


def _feature_path_for_id(piece: str, file_id: str) -> Path:
    """Construct absolute feature file path for a given piece + file id.

    The ids in queries.train.full are of the form
    'Chopin_Op017No4_Block-1995_pid9064-04', matching the .npy basename.
    """
    return MAZURKA_FEATURE_ROOT / piece / f"{file_id}.npy"

In [3]:
def _count_chunk_edge_positions(D_top, D_right):
    """Return number of unique edge positions with finite cost for one chunk.

    We treat the shared corner between top/right edges as a single position.
    """
    D_top = np.asarray(D_top, dtype=float)
    D_right = np.asarray(D_right, dtype=float)

    finite_top = np.isfinite(D_top)
    finite_right = np.isfinite(D_right)

    num = int(finite_top.sum() + finite_right.sum())
    if D_top.size > 0 and D_right.size > 0:
        if np.isfinite(D_top[-1]) and np.isfinite(D_right[-1]):
            num -= 1  # do not double-count the corner
    return max(num, 0)


def _count_chunk_border_slots(rows: int, cols: int) -> int:
    """Return number of unique border slots on top+right for a chunk."""
    rows = int(rows)
    cols = int(cols)
    if rows <= 0 or cols <= 0:
        return 0
    return rows + cols - 1


def compute_sparsity_stats_from_chunks(tiled_result, chunks_dict, D_chunks, L_chunks):
    """Compute sparsity statistics for one alignment.

    Stat 1 — endings over chunk cells:
        Sum over chunks of (finite edge ending positions on top+right, corner once)
        divided by sum of valid top+right border slots for each chunk using its
        actual shape (rows + cols - 1).

    Stat 2 — same ratio after excluding tile indices on the last row or last
        column of the chunk grid (``i == n_row - 1`` or ``j == n_col - 1``).
        Those tiles are the ones where Stage-1 fills every position on the
        top or right edge for the global boundary (see ``dp_fill_chunks`` in
        ``Sparse_Parflex.py``).
    """
    n_row = int(tiled_result["n_row"])
    n_col = int(tiled_result["n_col"])

    num_chunks = n_row * n_col

    total_endings = 0
    total_chunk_cells = 0

    total_endings_interior = 0
    total_chunk_cells_interior = 0
    n_interior_tiles = 0

    for i in range(n_row):
        for j in range(n_col):
            rows, cols = chunks_dict[(i, j)]["shape"]
            chunk_cells = _count_chunk_border_slots(rows, cols)

            D_top = D_chunks[i][j][0] if 0 in D_chunks[i][j] else []
            D_right = D_chunks[i][j][1] if 1 in D_chunks[i][j] else []

            num_valid = _count_chunk_edge_positions(D_top, D_right)
            total_endings += num_valid
            total_chunk_cells += chunk_cells

            is_global_edge_tile = (i == n_row - 1) or (j == n_col - 1)
            if not is_global_edge_tile:
                total_endings_interior += num_valid
                total_chunk_cells_interior += chunk_cells
                n_interior_tiles += 1

    ratio_all = total_endings / total_chunk_cells if total_chunk_cells > 0 else float("nan")
    if total_chunk_cells_interior > 0 and n_interior_tiles > 0:
        ratio_interior = total_endings_interior / total_chunk_cells_interior
    else:
        ratio_interior = float("nan")

    return {
        "endings_over_chunk_cells": float(ratio_all),
        "endings_over_chunk_cells_excluding_global_edge_tiles": float(ratio_interior),
        "total_endings_computed": int(total_endings),
        "total_chunk_cells": int(total_chunk_cells),
        "total_endings_interior": int(total_endings_interior),
        "total_chunk_cells_interior": int(total_chunk_cells_interior),
        "num_chunks": int(num_chunks),
        "n_interior_tiles": int(n_interior_tiles),
    }


def compute_alignment_sparsity(F1_path: Path, F2_path: Path, beta: float = 0.1):
    """Run sparse Parflex Stage-1/DP on one pair and return sparsity stats."""
    F1 = np.load(F1_path, allow_pickle=True)
    F2 = np.load(F2_path, allow_pickle=True)

    C, tiled_result = align_system_sparse_parflex(F1, F2, beta=beta, L=DEFAULT_CHUNK_LENGTH)

    chunks_dict = tiled_result["chunks_dict"]
    L_block = tiled_result["L_block"]
    n_row = tiled_result["n_row"]
    n_col = tiled_result["n_col"]

    D_chunks, L_chunks = chunked_flexdtw(chunks_dict, L_block, n_row, n_col, buffer_param=1)
    D_chunks, L_chunks = sync_overlapping_positions(D_chunks, L_chunks, n_row, n_col)

    stats = compute_sparsity_stats_from_chunks(tiled_result, chunks_dict, D_chunks, L_chunks)
    return stats

In [4]:
def compute_sparsity_for_mazurka(piece: str, max_pairs: int | None = None, beta: float = 0.1):
    """Compute sparsity stats for all (or first max_pairs) train pairs of one Mazurka piece."""
    piece_to_pairs = _parse_train_queries(TRAIN_QUERIES_PATH)
    if piece not in piece_to_pairs:
        raise ValueError(f"Piece {piece!r} not found in {TRAIN_QUERIES_PATH}")

    pairs = piece_to_pairs[piece]
    if max_pairs is not None:
        pairs = pairs[:max_pairs]

    per_pair_stats = []

    for (fileid1, fileid2) in pairs:
        f1 = _feature_path_for_id(piece, fileid1)
        f2 = _feature_path_for_id(piece, fileid2)
        stats = compute_alignment_sparsity(f1, f2, beta=beta)
        per_pair_stats.append(stats)

    if not per_pair_stats:
        raise ValueError(f"No pairs found for piece {piece!r}")

    r_all = np.array([s["endings_over_chunk_cells"] for s in per_pair_stats], dtype=float)
    r_int = np.array(
        [s["endings_over_chunk_cells_excluding_global_edge_tiles"] for s in per_pair_stats],
        dtype=float,
    )

    summary = {
        "piece": piece,
        "n_pairs": len(per_pair_stats),
        "mean_endings_over_chunk_cells": float(np.nanmean(r_all)),
        "std_endings_over_chunk_cells": float(np.nanstd(r_all, ddof=0)),
        "mean_endings_over_chunk_cells_excluding_global_edge_tiles": float(np.nanmean(r_int)),
        "std_endings_over_chunk_cells_excluding_global_edge_tiles": float(np.nanstd(r_int, ddof=0)),
    }

    return summary, per_pair_stats


# --- Driver: compute and print summary for train Mazurka #1 and #2 ---

mazurka1_piece = TRAIN_MAZURKAS["Mazurka_1"]
mazurka2_piece = TRAIN_MAZURKAS["Mazurka_2"]

summary_1, per_pair_1 = compute_sparsity_for_mazurka(mazurka1_piece)
summary_2, per_pair_2 = compute_sparsity_for_mazurka(mazurka2_piece)

print("Train Mazurka #1 (", mazurka1_piece, ")")
print("  Endings / chunk cells (all tiles):              ", summary_1["mean_endings_over_chunk_cells"])
print("  Endings / chunk cells (exclude last row/col):   ", summary_1["mean_endings_over_chunk_cells_excluding_global_edge_tiles"])
print("  # alignment pairs: ", summary_1["n_pairs"])
print()
print("Train Mazurka #2 (", mazurka2_piece, ")")
print("  Endings / chunk cells (all tiles):              ", summary_2["mean_endings_over_chunk_cells"])
print("  Endings / chunk cells (exclude last row/col): ", summary_2["mean_endings_over_chunk_cells_excluding_global_edge_tiles"])
print("  # alignment pairs: ", summary_2["n_pairs"])

Train Mazurka #1 ( Chopin_Op017No4 )
  Endings / chunk cells (all tiles):               0.3451922057630328
  Endings / chunk cells (exclude last row/col):    0.02910622884440176
  # alignment pairs:  1953

Train Mazurka #2 ( Chopin_Op063No3 )
  Endings / chunk cells (all tiles):               0.5206399704468392
  Endings / chunk cells (exclude last row/col):  0.021399259285676122
  # alignment pairs:  3828
